In [2]:
# Run this ONLY if Cell 5 keeps getting stuck at the same %
!rm -rf /root/.cache/huggingface/hub/models--meta-llama--Llama-3.2-1B-Instruct
print("Cache cleared. Try running the cells again.")

Cache cleared. Try running the cells again.


In [3]:

import torch
import gc
import re
from tqdm import tqdm
from torch.utils.data import DataLoader
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoModelForSequenceClassification
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
from datasets import load_dataset
from huggingface_hub import login



In [ ]:
# 2. LOGIN HERE (Required for Llama 3.2)
# Replace 'your_token_here' with your actual HF token string

HF_TOKEN = "your token here" 
login(token=HF_TOKEN)



In [5]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [6]:
#cell 2
def parse_hh_rlhf(example):
    """
    Parses 'chosen' and 'rejected' strings into (prompt, chosen, rejected) triples.
    Expected format: "\n\nHuman: ... \n\nAssistant: ..."
    """
    def split_text(text):
        # We split at the last instance of 'Assistant: ' to get the response
        parts = text.rsplit('\n\nAssistant: ', 1)
        if len(parts) < 2:
            return text, ""
        prompt = parts[0] + '\n\nAssistant: '
        response = parts[1]
        return prompt, response

    # Extract prompt and response for both chosen and rejected paths
    prompt, chosen_resp = split_text(example['chosen'])
    _, rejected_resp = split_text(example['rejected'])
    
    return {
        "prompt": prompt,
        "chosen": chosen_resp,
        "rejected": rejected_resp
    }

# Load using the 'harmless-base' directory specifically
print("Loading 'harmless-base' dataset...")
try:
    dataset = load_dataset("Anthropic/hh-rlhf", data_dir="harmless-base", split="train")
except Exception as e:
    print(f"Error loading with data_dir: {e}")
    print("Attempting to load default split...")
    dataset = load_dataset("Anthropic/hh-rlhf", split="train")

# Apply the parsing function
dataset = dataset.map(parse_hh_rlhf)

# Split into train/val as per instructions
split_dataset = dataset.train_test_split(test_size=0.1, seed=42)
train_data = split_dataset['train']
val_data = split_dataset['test']

print(f"\nDataset Loaded Successfully!")
print(f"Train size: {len(train_data)} | Val size: {len(val_data)}")
print("-" * 30)
print(f"Sample Prompt:\n{train_data[0]['prompt'][:100]}...")
print(f"\nSample Chosen Response:\n{train_data[0]['chosen'][:100]}...")

Loading 'harmless-base' dataset...


README.md: 0.00B [00:00, ?B/s]

harmless-base/train.jsonl.gz:   0%|          | 0.00/13.2M [00:00<?, ?B/s]

harmless-base/test.jsonl.gz:   0%|          | 0.00/743k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/42537 [00:00<?, ? examples/s]


Dataset Loaded Successfully!
Train size: 38283 | Val size: 4254
------------------------------
Sample Prompt:


Human: I just want to slap my friend right now.

Assistant: ...

Sample Chosen Response:
OK, so why don’t you do that?...


In [7]:
# New Cell: DataLoaders for SFT, RM, and DPO starrr after added missing
from torch.utils.data import DataLoader

def collate_sft(batch):
    return [item['prompt'] + item['chosen'] for item in batch]

def collate_rm(batch):
    # Returns pairs of (chosen_full_text, rejected_full_text)
    return [(item['prompt'] + item['chosen'], item['prompt'] + item['rejected']) for item in batch]

def collate_dpo(batch):
    # Returns raw triples for the DPO trainer to handle
    return [{"prompt": item['prompt'], "chosen": item['chosen'], "rejected": item['rejected']} for item in batch]

# Create the loaders (using a small batch size for safety)
train_subset = train_data.select(range(1000)) # Using subset for speed

loader_sft = DataLoader(train_subset, batch_size=4, collate_fn=collate_sft)
loader_rm  = DataLoader(train_subset, batch_size=4, collate_fn=collate_rm)
loader_dpo = DataLoader(train_subset, batch_size=4, collate_fn=collate_dpo)

print(f"✅ Created 3 Dataloaders: SFT, RM, and DPO.")

✅ Created 3 Dataloaders: SFT, RM, and DPO.


In [ ]:

import json
import os

In [11]:
REWARD_ID = "/kaggle/input/datasets/karan943/meta-llamallama-3-2-1b-instruct/meta-llama/Llama-3.2-1B-Instruct"

In [12]:
with open(os.path.join(REWARD_ID, "config.json"), "r") as f:
    config = json.load(f)

print(f"Model: {config.get('model_type')}")
print(f"Hidden Size: {config.get('hidden_size')} (Expected: 2048)")
print(f"Layers: {config.get('num_hidden_layers')} (Expected: 16)")

if config.get('hidden_size') == 2048:
    print("\n✅ VERIFIED: This is the correct 1B model. Let's start!")
else:
    print("\n❌ STOP: This is not the 1B model.")

Model: llama
Hidden Size: 2048 (Expected: 2048)
Layers: 16 (Expected: 16)

✅ VERIFIED: This is the correct 1B model. Let's start!


In [14]:
# Cell 3 starr fourth update
# Cell 3: Robust Model Loading Utility
# Cell 3: Simplified Loader (No 4-bit required for 1B model)
def load_base_model(model_id, is_reward=False):
    print(f"Loading {model_id}...")
    
    tokenizer = AutoTokenizer.from_pretrained(model_id, token=HF_TOKEN, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"
    
    # We use bfloat16 directly. It's faster and avoids bitsandbytes errors.
    # 1B model in bfloat16 = 2GB. Kaggle has 15GB. We are safe!
    model_kwargs = {
        "device_map": "auto",
        "torch_dtype": torch.bfloat16, 
        "token": HF_TOKEN,
        "trust_remote_code": True,
        "low_cpu_mem_usage": True
    }
    
    if is_reward:
        model = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=1, **model_kwargs)
    else:
        model = AutoModelForCausalLM.from_pretrained(model_id, **model_kwargs)
    
    # Gradient checkpointing saves VRAM during training
    model.gradient_checkpointing_enable()
    return model, tokenizer

POLICY_ID = "HuggingFaceTB/SmolLM2-1.7B-Instruct" 
REWARD_ID = "/kaggle/input/datasets/karan943/meta-llamallama-3-2-1b-instruct/meta-llama/Llama-3.2-1B-Instruct"

In [ ]:
#cell 4
def apply_lora(model, task_type=TaskType.CAUSAL_LM):
    lora_config = LoraConfig(
        r=8,
        lora_alpha=16,
        target_modules=["q_proj", "v_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type=task_type
    )
    return get_peft_model(model, lora_config)

In [28]:
# Cell 5: Load Reward Model (Using the function from Cell 3)  double starr
# Make sure REWARD_ID is set to that long path we found earlier!

print(f"Loading Reward Model from: {REWARD_ID}")

# 1. Use the function we defined in Cell 3
rm_model, rm_tokenizer = load_base_model(REWARD_ID, is_reward=True)

# 2. Apply LoRA (Task C1.1)
# Note: task_type must be SEQ_CLS for the Reward Model
rm_model = apply_lora(rm_model, task_type=TaskType.SEQ_CLS)

# 3. Final config check
rm_model.config.pad_token_id = rm_tokenizer.pad_token_id

print("✅ SUCCESS: Reward Model (Llama-3.2-1B) is loaded and ready!")

Loading Reward Model from: /kaggle/input/datasets/karan943/meta-llamallama-3-2-1b-instruct/meta-llama/Llama-3.2-1B-Instruct
Loading /kaggle/input/datasets/karan943/meta-llamallama-3-2-1b-instruct/meta-llama/Llama-3.2-1B-Instruct...


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

LlamaForSequenceClassification LOAD REPORT from: /kaggle/input/datasets/karan943/meta-llamallama-3-2-1b-instruct/meta-llama/Llama-3.2-1B-Instruct
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✅ SUCCESS: Reward Model (Llama-3.2-1B) is loaded and ready!


In [29]:
# Cell 6: Train the Reward Model (Task C1.2)
import torch.nn.functional as F

In [30]:
# 1. Setup Optimizer (Lower learning rate for stability)
optimizer_rm = torch.optim.AdamW(rm_model.parameters(), lr=5e-5)
rm_model.train()

# 2. Training for 1,000 samples (as a representative subset for Kaggle)
# Following the Margin Ranking Loss from Task C1.2
for i, batch in enumerate(tqdm(train_data.select(range(1000)), desc="RM Training")):
    # Step A: Prepare text pairs (Task C1.2, item 3)
    c_text = batch['prompt'] + batch['chosen']
    r_text = batch['prompt'] + batch['rejected']
    
    # Step B: Tokenize (Task C1.2, item 4)
    c_in = rm_tokenizer(c_text, return_tensors="pt", truncation=True, max_length=512).to(device)
    r_in = rm_tokenizer(r_text, return_tensors="pt", truncation=True, max_length=512).to(device)
    
    # Step C: Forward Pass
    c_logits = rm_model(**c_in).logits # Reward for chosen
    r_logits = rm_model(**r_in).logits # Reward for rejected
    
    # Step D: Margin Ranking Loss + Regularization (Task C1.2, item 5)
    # L = -log(sigmoid(r_chosen - r_rejected)) + reg * (r_chosen^2 + r_rejected^2)
    loss = -F.logsigmoid(c_logits - r_logits).mean() 
    reg_loss = 0.001 * (c_logits**2 + r_logits**2).mean() # lambda_reg = 10^-3
    
    total_loss = loss + reg_loss
    
    # Step E: Backward Pass
    total_loss.backward()
    optimizer_rm.step()
    optimizer_rm.zero_grad()
    
    # Log every 100 steps
    if i % 100 == 0:
        accuracy = (c_logits > r_logits).float().mean().item()
        print(f"Step {i} | Loss: {total_loss.item():.4f} | Accuracy: {accuracy*100}%")

# 3. Save the Reward Model weights (Task C1.3, item 9)
rm_model.save_pretrained("reward_model_lora")
print("\n✅ Task C1 Complete: Reward Model trained and saved!")

RM Training:   0%|          | 1/1000 [00:02<36:19,  2.18s/it]

Step 0 | Loss: 0.3887 | Accuracy: 100.0%


RM Training:  10%|█         | 101/1000 [01:18<16:05,  1.07s/it]

Step 100 | Loss: 0.7070 | Accuracy: 0.0%


RM Training:  20%|██        | 201/1000 [02:39<13:11,  1.01it/s]

...

Step 800 | Loss: 0.4590 | Accuracy: 100.0%


RM Training:  90%|█████████ | 901/1000 [12:45<01:29,  1.11it/s]

Step 900 | Loss: 0.6172 | Accuracy: 100.0%


RM Training: 100%|██████████| 1000/1000 [14:05<00:00,  1.18it/s]


✅ Task C1 Complete: Reward Model trained and saved!


In [ ]:
# Cell 7: PURGE RM
del rm_model, optimizer_rm
gc.collect(); torch.cuda.empty_cache()

In [ ]:
# checking model type etc
CHECK_PATH = "/kaggle/input/models/kawchar85/smollm2-1.7b-instruct/transformers/default_unsloth/1" 

print(f"Checking path: {CHECK_PATH}")

try:
    with open(os.path.join(CHECK_PATH, "config.json"), "r") as f:
        cfg = json.load(f)
    
    print(f"\n--- Model Identity Check ---")
    print(f"Model Type: {cfg.get('model_type')}")         # Must be 'llama'
    print(f"Hidden Size: {cfg.get('hidden_size')}")       # Must be 2048
    print(f"Layers: {cfg.get('num_hidden_layers')}")     # Must be 24
    
    if cfg.get('num_hidden_layers') == 24:
        print("\n✅ VERIFIED: This is the exact SmolLM2-1.7B-Instruct model.")
        print("You can now proceed to SFT Training (Cell 8).")
    else:
        print("\n❌ ERROR: This is a different model. Check the path.")

except Exception as e:
    print(f"❌ ERROR: Could not find files. Did you click 'Copy Path' on the folder? {e}")

Checking path: /kaggle/input/models/kawchar85/smollm2-1.7b-instruct/transformers/default_unsloth/1

--- Model Identity Check ---
Model Type: llama
Hidden Size: 2048
Layers: 24

✅ VERIFIED: This is the exact SmolLM2-1.7B-Instruct model.
You can now proceed to SFT Training (Cell 8).


In [21]:
POLICY_ID = "/kaggle/input/models/kawchar85/smollm2-1.7b-instruct/transformers/default_unsloth/1"

In [22]:
# Cell 8: Train SFT Policy (Task C2)
print(f"Loading SFT Policy from: {POLICY_ID}")

# 1. Load the model and tokenizer from the verified local path
sft_model, sft_tokenizer = load_base_model(POLICY_ID)
sft_model = apply_lora(sft_model)

# 2. Setup Optimizer
optimizer_sft = torch.optim.AdamW(sft_model.parameters(), lr=2e-5)
sft_model.train()

# 3. Training Loop (1,000 steps as per our setup)
for i, batch in enumerate(tqdm(train_data.select(range(1000)), desc="SFT Training")):
    # Step A: Prepare inputs
    full_text = batch['prompt'] + batch['chosen']
    inputs = sft_tokenizer(full_text, return_tensors="pt", truncation=True, max_length=512).to(device)
    
    # Step B: Label masking (Task C2.1 - only compute loss on response tokens)
    labels = inputs.input_ids.clone()
    prompt_ids = sft_tokenizer(batch['prompt'], return_tensors="pt")['input_ids']
    # Mask prompt tokens with -100
    labels[:, :prompt_ids.shape[1]] = -100 
    
    # Step C: Forward & Backward
    loss = sft_model(**inputs, labels=labels).loss
    loss.backward()
    
    optimizer_sft.step()
    optimizer_sft.zero_grad()
    
    if i % 100 == 0:
        print(f"Step {i} | Loss: {loss.item():.4f}")

# 4. Save the SFT adapters (Task C2.3)
sft_model.save_pretrained("sft_policy_lora")
print("\n✅ Task C2 Complete: SFT Policy saved successfully!")


Loading SFT Policy from: /kaggle/input/models/kawchar85/smollm2-1.7b-instruct/transformers/default_unsloth/1
Loading /kaggle/input/models/kawchar85/smollm2-1.7b-instruct/transformers/default_unsloth/1...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

SFT Training:   0%|          | 1/1000 [00:02<35:33,  2.14s/it]

Step 0 | Loss: 2.2857


...

Step 800 | Loss: 1.3877


SFT Training:  90%|█████████ | 901/1000 [11:47<01:26,  1.15it/s]

Step 900 | Loss: 1.5027


SFT Training: 100%|██████████| 1000/1000 [13:04<00:00,  1.27it/s]


✅ Task C2 Complete: SFT Policy saved successfully!


In [23]:
# Cell 9: PURGE SFT
del sft_model, optimizer_sft
gc.collect(); torch.cuda.empty_cache()

In [15]:
# Cell 10: PPO Model Loading (Task C3.1) starrr 22 
from peft import PeftModel

print("Loading 4 models for PPO... this will take a few minutes.")

# A. The Trainable Policy (starts from SFT)
base_p, p_tok = load_base_model(POLICY_ID)
policy = PeftModel.from_pretrained(base_p, "sft_policy_lora", is_trainable=True)

# B. The Frozen Reference (SFT anchor)
base_ref, _ = load_base_model(POLICY_ID)
ref_model = PeftModel.from_pretrained(base_ref, "sft_policy_lora")
ref_model.eval()

# C. The Frozen Reward Model
base_rm, rm_tok = load_base_model(REWARD_ID, is_reward=True)
reward_model = PeftModel.from_pretrained(base_rm, "reward_model_lora")
reward_model.eval()

# D. The Value Head (Critic) - Task C3.1
class ValueHead(torch.nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.head = torch.nn.Linear(hidden_size, 1)
        torch.nn.init.normal_(self.head.weight, std=0.01)
    def forward(self, hs): return self.head(hs)

v_head = ValueHead(policy.config.hidden_size).to(device).to(torch.bfloat16)

print("✅ All models loaded. Current VRAM usage: check your sidebar!")

Loading 4 models for PPO... this will take a few minutes.
Loading HuggingFaceTB/SmolLM2-1.7B-Instruct...


config.json:   0%|          | 0.00/908 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

Loading /kaggle/input/datasets/karan943/meta-llamallama-3-2-1b-instruct/meta-llama/Llama-3.2-1B-Instruct...


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

LlamaForSequenceClassification LOAD REPORT from: /kaggle/input/datasets/karan943/meta-llamallama-3-2-1b-instruct/meta-llama/Llama-3.2-1B-Instruct
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✅ All models loaded. Current VRAM usage: check your sidebar!


In [16]:
# PPO Tokenizer Fix
reward_model.config.pad_token_id = rm_tok.pad_token_id
policy.config.pad_token_id = p_tok.pad_token_id
ref_model.config.pad_token_id = p_tok.pad_token_id

print(f"Reward Model Pad Token ID: {reward_model.config.pad_token_id}")
print("✅ Padding IDs fixed. You can now run the Training Loop.")

Reward Model Pad Token ID: 128009
✅ Padding IDs fixed. You can now run the Training Loop.


In [16]:
# Cell 11 & 12: Fixed DType PPO Training Loop
import torch.nn.functional as F

# 1. Setup Optimizer (Task C3.1)
optimizer_ppo = torch.optim.AdamW(list(policy.parameters()) + list(v_head.parameters()), lr=5e-7)

# 2. Advantage Utility
def compute_advantages(rewards, values, gamma=1.0, lam=0.95):
    advantages = torch.zeros_like(rewards)
    last_gae = 0
    # Ensure all math is done in Float32 to avoid precision errors
    rewards_f = rewards.float()
    values_f = values.float()
    for t in reversed(range(rewards.size(1))):
        next_val = values_f[:, t+1] if t < rewards.size(1)-1 else 0
        delta = rewards_f[:, t] + gamma * next_val - values_f[:, t]
        advantages[:, t] = last_gae = delta + gamma * lam * last_gae
    # Standardize
    return (advantages - advantages.mean()) / (advantages.std() + 1e-8)

# 3. Training Loop
for step in range(200):
    batch = train_data.select(range(step*2, (step*2) + 2)) 
    prompts = [b['prompt'] for b in batch]
    
    # A. ROLLOUT (Task C3.2)
    inputs = p_tok(prompts, return_tensors="pt", padding=True).to(device)
    policy.eval()
    with torch.no_grad():
        outputs = policy.generate(
            **inputs, 
            max_new_tokens=32, 
            do_sample=True, 
            temperature=0.7,
            pad_token_id=p_tok.pad_token_id
        )
    
    res_ids = outputs[:, inputs.input_ids.shape[1]:]
    full_seq = outputs
    
    # B. SCORE
    with torch.no_grad():
        # Policy logprobs - kept in bfloat16 for the forward pass
        logits = policy(full_seq).logits[:, inputs.input_ids.shape[1]-1:-1, :]
        pi_lp = torch.gather(torch.log_softmax(logits.float(), dim=-1), -1, res_ids.unsqueeze(-1)).squeeze(-1)
        
        # Reference logprobs
        ref_logits = ref_model(full_seq).logits[:, inputs.input_ids.shape[1]-1:-1, :]
        ref_lp = torch.gather(torch.log_softmax(ref_logits.float(), dim=-1), -1, res_ids.unsqueeze(-1)).squeeze(-1)
        
        # RM Score
        texts = p_tok.batch_decode(full_seq, skip_special_tokens=True)
        rm_in = rm_tok(texts, return_tensors="pt", padding=True, truncation=True).to(device)
        task_rew = reward_model(**rm_in).logits.squeeze(-1)

    # C. REWARDS & ADVANTAGES (Task C3.3)
    kl = pi_lp - ref_lp # Still in Float32
    rewards_matrix = -0.1 * kl # beta = 0.1
    for b in range(len(prompts)):
        non_pad = (res_ids[b] != p_tok.pad_token_id).nonzero()
        last_idx = non_pad[-1].item() if len(non_pad) > 0 else 0
        rewards_matrix[b, last_idx] += task_rew[b].float() # Explicitly float
    
    with torch.no_grad():
        # Hidden states come out as BFloat16
        hs = policy(full_seq, output_hidden_states=True).hidden_states[-1][:, inputs.input_ids.shape[1]-1:-1, :]
        # Pass BFloat16 to the BFloat16 v_head (Fixing the previous error)
        values = v_head(hs).squeeze(-1)
    
    # compute_advantages handles the float conversion internally
    advantages = compute_advantages(rewards_matrix, values).to(device)
    
    # D. UPDATE (Task C3.4)
    policy.train(); v_head.train()
    optimizer_ppo.zero_grad()
    
    new_logits = policy(full_seq).logits[:, inputs.input_ids.shape[1]-1:-1, :]
    new_lp = torch.gather(torch.log_softmax(new_logits.float(), dim=-1), -1, res_ids.unsqueeze(-1)).squeeze(-1)
    
    # Math in Float32 for stability
    ratio = torch.exp(torch.clamp(new_lp - pi_lp.float(), -10, 10)) 
    surr1 = ratio * advantages.float()
    surr2 = torch.clamp(ratio, 0.8, 1.2) * advantages.float()
    
    loss = -torch.min(surr1, surr2).mean()
    loss.backward()
    
    # Gradient clipping
    torch.nn.utils.clip_grad_norm_(policy.parameters(), 1.0)
    optimizer_ppo.step()
    
    if step % 10 == 0:
        print(f"Step {step} | Loss: {loss.item():.4f} | Reward: {task_rew.mean().item():.2f}")

policy.save_pretrained("ppo_policy_lora")
print("✅ PPO Training Complete!")

In [19]:
# Cell 13: Ultra PPO PURGE
# 1. Delete all high-level model objects
try:
    del policy, ref_model, reward_model, v_head, optimizer_ppo
except NameError:
    pass

# 2. Delete any "base" models if they exist in memory
try:
    del base_p, base_ref, base_rm
except NameError:
    pass

# 3. Force garbage collection and clear CUDA cache
gc.collect()
torch.cuda.empty_cache()

print("✅ VRAM completely cleared. Memory is fresh for DPO/GRPO.")

✅ VRAM completely cleared. Memory is fresh for DPO/GRPO.


In [20]:
# Cell 14: DPO Setup
print("Setting up DPO models...")

# 1. Load the Base Policy and the SFT adapters as the starting point
base_dpo, tokenizer_dpo = load_base_model(POLICY_ID)
dpo_model = PeftModel.from_pretrained(base_dpo, "sft_policy_lora", is_trainable=True)

# 2. Load the Reference Model (Frozen version of SFT)
ref_dpo, _ = load_base_model(POLICY_ID)
ref_dpo = PeftModel.from_pretrained(ref_dpo, "sft_policy_lora")
ref_dpo.eval()

optimizer_dpo = torch.optim.AdamW(dpo_model.parameters(), lr=5e-7)

print("✅ DPO Models ready. Memory check: You should have plenty of space.")

Setting up DPO models...
Loading HuggingFaceTB/SmolLM2-1.7B-Instruct...


Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

Loading HuggingFaceTB/SmolLM2-1.7B-Instruct...


Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

✅ DPO Models ready. Memory check: You should have plenty of space.


In [21]:
# Cell 15: DPO Training Loop (Task C4.1 - C4.3)
import torch.nn.functional as F

def get_logprobs(model, prompts, responses):
    """Computes log-probs for the response tokens only (Task C4.1)"""
    texts = [p + r for p, r in zip(prompts, responses)]
    inputs = tokenizer_dpo(texts, return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)
    
    # Forward pass
    logits = model(**inputs).logits[:, :-1, :] # Shift for prediction
    labels = inputs.input_ids[:, 1:]
    
    # Calculate per-token log-probs
    log_softmax_logits = F.log_softmax(logits.float(), dim=-1)
    per_token_logps = torch.gather(log_softmax_logits, -1, labels.unsqueeze(-1)).squeeze(-1)
    
    # Mask out prompt tokens (Task C4.1 item 1)
    # We find where the response starts by tokenizing the prompts separately
    prompt_inputs = tokenizer_dpo(prompts, return_tensors="pt", padding=True)
    prompt_len = prompt_inputs.input_ids.shape[1]
    
    mask = torch.ones_like(per_token_logps)
    mask[:, :prompt_len-1] = 0 # Mask prompt tokens
    mask[labels == tokenizer_dpo.pad_token_id] = 0 # Mask padding
    
    return (per_token_logps * mask).sum(-1)

# Training Loop (1 Epoch on a subset of 400 pairs for Kaggle)
dpo_model.train()
beta_dpo = 0.1 # coefficient from manual Task 4.2

for i, batch in enumerate(tqdm(train_data.select(range(400)), desc="DPO Training")):
    # 1. Get logprobs for Policy and Reference
    with torch.no_grad():
        ref_lp_w = get_logprobs(ref_dpo, [batch['prompt']], [batch['chosen']])
        ref_lp_l = get_logprobs(ref_dpo, [batch['prompt']], [batch['rejected']])
        
    pi_lp_w = get_logprobs(dpo_model, [batch['prompt']], [batch['chosen']])
    pi_lp_l = get_logprobs(dpo_model, [batch['prompt']], [batch['rejected']])
    
    # 2. Compute DPO Loss (Task 4.2, Equation 10)
    # loss = -log_sigmoid(beta * ((pi_w - ref_w) - (pi_l - ref_l)))
    logits = beta_dpo * ((pi_lp_w - ref_lp_w) - (pi_lp_l - ref_lp_l))
    loss = -F.logsigmoid(logits).mean()
    
    # 3. Step
    loss.backward()
    optimizer_dpo.step()
    optimizer_dpo.zero_grad()
    
    if i % 100 == 0:
        print(f"DPO Step {i} | Loss: {loss.item():.4f}")

dpo_model.save_pretrained("dpo_policy_lora")
print("✅ DPO Saved!")

DPO Training:   0%|          | 1/400 [00:01<08:01,  1.21s/it]

DPO Step 0 | Loss: 0.7051


DPO Training:  25%|██▌       | 101/400 [03:12<11:34,  2.32s/it]

DPO Step 100 | Loss: 0.6676


DPO Training:  50%|█████     | 201/400 [06:32<08:25,  2.54s/it]

DPO Step 200 | Loss: 0.6937


DPO Training:  75%|███████▌  | 301/400 [10:13<04:20,  2.63s/it]

DPO Step 300 | Loss: 0.7264


DPO Training: 100%|██████████| 400/400 [13:50<00:00,  2.08s/it]
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:309: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(


✅ DPO Saved!


In [22]:
# Cell 15.5: DPO PURGE
try:
    del dpo_model, ref_dpo, base_dpo, optimizer_dpo
except: pass
gc.collect(); torch.cuda.empty_cache()
print("✅ VRAM Purged. Ready for GRPO.")

✅ VRAM Purged. Ready for GRPO.


In [24]:
# Cell 16: GRPO Setup (Task C5)
print("Setting up GRPO models...")

# 1. Load the Policy (starting from SFT)
base_grpo, _ = load_base_model(POLICY_ID)
grpo_model = PeftModel.from_pretrained(base_grpo, "sft_policy_lora", is_trainable=True)

# 2. Load the Reference Model (Frozen SFT)
base_ref_g, _ = load_base_model(POLICY_ID)
ref_grpo = PeftModel.from_pretrained(base_ref_g, "sft_policy_lora")
ref_grpo.eval()

# 3. Load the Reward Model (Frozen Llama)
base_rm_g, rm_tok_g = load_base_model(REWARD_ID, is_reward=True)
rm_grpo = PeftModel.from_pretrained(base_rm_g, "reward_model_lora")
rm_grpo.eval()
rm_grpo.config.pad_token_id = rm_tok_g.pad_token_id

optimizer_grpo = torch.optim.AdamW(grpo_model.parameters(), lr=5e-7)

print("✅ GRPO Models loaded. Ready for Group Rollout logic!")

Setting up GRPO models...
Loading HuggingFaceTB/SmolLM2-1.7B-Instruct...


Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

Loading HuggingFaceTB/SmolLM2-1.7B-Instruct...


Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

Loading /kaggle/input/datasets/karan943/meta-llamallama-3-2-1b-instruct/meta-llama/Llama-3.2-1B-Instruct...


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

LlamaForSequenceClassification LOAD REPORT from: /kaggle/input/datasets/karan943/meta-llamallama-3-2-1b-instruct/meta-llama/Llama-3.2-1B-Instruct
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✅ GRPO Models loaded. Ready for Group Rollout logic!


In [25]:
# Cell 17: GRPO Training Loop (Task C5.1 - C5.2)
# Parameters from manual: K=4, beta=0.1, eps=0.2
K_GROUP = 4
BETA_GRPO = 0.1
EPS_GRPO = 0.2

for step in range(100): # GRPO is computationally heavy; 100 steps for Kaggle
    batch = train_data.select(range(step, step + 1)) # 1 prompt at a time
    prompt = batch[0]['prompt']
    
    # 1. Group Rollout: Sample K completions (Task C5.1)
    inputs = tokenizer_dpo([prompt] * K_GROUP, return_tensors="pt").to(device)
    grpo_model.eval()
    with torch.no_grad():
        outputs = grpo_model.generate(
            **inputs, 
            max_new_tokens=48, 
            do_sample=True, 
            temperature=0.9,
            pad_token_id=tokenizer_dpo.pad_token_id
        )
    
    res_ids = outputs[:, inputs.input_ids.shape[1]:]
    full_seq = outputs
    
    # 2. Get Group Rewards & Relative Advantages (Task C5.1)
    with torch.no_grad():
        # Score completions with RM
        texts = tokenizer_dpo.batch_decode(full_seq, skip_special_tokens=True)
        rm_inputs = rm_tok_g(texts, return_tensors="pt", padding=True, truncation=True).to(device)
        rewards = rm_grpo(**rm_inputs).logits.squeeze(-1) # shape: (K,)
        
        # Group Advantage: (r - mean) / std (Equation from Task C5.1)
        mean_r = rewards.mean()
        std_r = rewards.std() + 1e-8
        advantages = (rewards - mean_r) / std_r
        
        # Get 'Old' Log-probs for the ratio
        logits_old = grpo_model(full_seq).logits[:, inputs.input_ids.shape[1]-1:-1, :]
        pi_lp_old = torch.gather(F.log_softmax(logits_old.float(), dim=-1), -1, res_ids.unsqueeze(-1)).squeeze(-1)
        
        # Get Reference Log-probs for KL penalty (Task C5.2 item 7)
        ref_logits = ref_grpo(full_seq).logits[:, inputs.input_ids.shape[1]-1:-1, :]
        ref_lp = torch.gather(F.log_softmax(ref_logits.float(), dim=-1), -1, res_ids.unsqueeze(-1)).squeeze(-1)

    # 3. GRPO Update (Task C5.2)
    grpo_model.train()
    optimizer_grpo.zero_grad()
    
    logits_new = grpo_model(full_seq).logits[:, inputs.input_ids.shape[1]-1:-1, :]
    pi_lp_new = torch.gather(F.log_softmax(logits_new.float(), dim=-1), -1, res_ids.unsqueeze(-1)).squeeze(-1)
    
    # Ratio and Clipping (Equation 12)
    ratio = torch.exp(torch.clamp(pi_lp_new - pi_lp_old, -10, 10))
    surr1 = ratio * advantages.unsqueeze(-1)
    surr2 = torch.clamp(ratio, 1.0 - EPS_GRPO, 1.0 + EPS_GRPO) * advantages.unsqueeze(-1)
    
    # KL Penalty (Approximate per-token KL)
    kl_penalty = (torch.exp(ref_lp - pi_lp_new) - (ref_lp - pi_lp_new) - 1)
    
    loss = -(torch.min(surr1, surr2) - BETA_GRPO * kl_penalty).mean()
    
    loss.backward()
    torch.nn.utils.clip_grad_norm_(grpo_model.parameters(), 1.0)
    optimizer_grpo.step()
    
    if step % 10 == 0:
        print(f"GRPO Step {step} | Mean Reward: {mean_r.item():.4f} | Loss: {loss.item():.4f}")

grpo_model.save_pretrained("grpo_policy_lora")
print("✅ GRPO Saved!")

GRPO Step 0 | Mean Reward: 0.7969 | Loss: -0.0048
GRPO Step 10 | Mean Reward: -0.9805 | Loss: 0.0011
GRPO Step 20 | Mean Reward: 0.7344 | Loss: 0.0027
GRPO Step 30 | Mean Reward: 0.7344 | Loss: -0.0006
GRPO Step 40 | Mean Reward: 0.8477 | Loss: -0.0005
GRPO Step 50 | Mean Reward: 0.0254 | Loss: 0.0003
GRPO Step 60 | Mean Reward: -0.5352 | Loss: -0.0065
GRPO Step 70 | Mean Reward: -0.0532 | Loss: 0.0034
GRPO Step 80 | Mean Reward: 1.5391 | Loss: 0.0011
GRPO Step 90 | Mean Reward: 0.1357 | Loss: 0.0013


/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:309: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(


✅ GRPO Saved!


In [26]:
# Cell 17.5: GRPO PURGE
try:
    del grpo_model, ref_grpo, rm_grpo, base_grpo, optimizer_grpo
except: pass
gc.collect(); torch.cuda.empty_cache()
print("✅ VRAM Purged. Ready for the final task: RLVR.")

✅ VRAM Purged. Ready for the final task: RLVR.


In [17]:
# Cell 18: RLVR Data & Answer Extraction logic (Task C6.1)
from datasets import load_dataset as load_ds
import re

# 1. Load GSM8K dataset (Task C6.1 item 1)
gsm_dataset = load_ds("openai/gsm8k", "main", split="train")

def extract_answer(text):
    """Robust answer extractor for GSM8K (Task C6.1 item 2)"""
    # Look for "#### [number]" or the last number in the string
    if "####" in text:
        res = text.split("####")[-1].strip()
    else:
        # Regex for numbers including decimals/commas
        nums = re.findall(r"[-+]?\d*\.\d+|\d+", text)
        res = nums[-1] if nums else None
    
    if res:
        res = res.replace(",", "")
        try: return float(res)
        except: return None
    return None

def math_reward_fn(generated_text, ground_truth):
    """Binary verifiable reward (Task C6.1 item 3)"""
    pred = extract_answer(generated_text)
    target = extract_answer(ground_truth)
    return 1.0 if (pred is not None and pred == target) else 0.0

print("✅ RLVR logic defined. Math problems loaded.")

README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

✅ RLVR logic defined. Math problems loaded.


In [19]:
# Import Refresh Cell
from peft import PeftModel, LoraConfig, get_peft_model, TaskType


In [20]:
import torch
import torch.nn.functional as F
from tqdm import tqdm
import re

print("✅ Imports refreshed. PeftModel is now defined.")

✅ Imports refreshed. PeftModel is now defined.


In [22]:
# Cell 19: Refined RLVR Training Loop
print(f"Loading SFT for RLVR starting point...")
base_rlvr, tok_rlvr = load_base_model(POLICY_ID)
rlvr_model = PeftModel.from_pretrained(base_rlvr, "sft_policy_lora", is_trainable=True)

optimizer_rlvr = torch.optim.AdamW(rlvr_model.parameters(), lr=1e-6)

for step in range(100):
    # 1. Get Math Problem
    example = gsm_dataset[step]
    
    # REFINED PROMPT: Forcing Chain-of-Thought as suggested in Section 2.3.1 (RLVR/DeepSeek-R1)
    prompt = f"Human: Solve the following math problem step by step. Show your reasoning clearly. At the end, provide the final numeric answer after '####'.\n\nQuestion: {example['question']}\n\nAssistant: Let's think step by step."
    
    # 2. Group Rollout (K=4)
    inputs = tok_rlvr([prompt]*4, return_tensors="pt", padding=True).to(device)
    rlvr_model.eval()
    with torch.no_grad():
        outputs = rlvr_model.generate(
            **inputs, 
            max_new_tokens=160, # Increased for step-by-step reasoning
            do_sample=True, 
            temperature=0.5,   # Lowered for better focus
            pad_token_id=tok_rlvr.pad_token_id
        )
    
    responses = outputs[:, inputs.input_ids.shape[1]:]
    
    # 3. Compute Rewards (Math Accuracy)
    gen_texts = tok_rlvr.batch_decode(responses, skip_special_tokens=True)
    rewards = torch.tensor([math_reward_fn(t, example['answer']) for t in gen_texts]).to(device)
    
    # 4. Group Advantage Calculation
    mean_r = rewards.mean()
    # If all 4 are wrong, std is 0. We add epsilon to prevent division by zero.
    std_r = rewards.std() + 1e-8
    advantages = (rewards - mean_r) / std_r
    
    # 5. Policy Gradient Update
    rlvr_model.train()
    optimizer_rlvr.zero_grad()
    
    logits = rlvr_model(outputs).logits[:, inputs.input_ids.shape[1]-1:-1, :]
    logprobs = torch.gather(torch.log_softmax(logits.float(), dim=-1), -1, responses.unsqueeze(-1)).squeeze(-1)
    
    # Loss = -log_prob * advantage
    loss = -(logprobs.sum(dim=-1) * advantages).mean()
    
    # 6. Step (Only if at least one answer in the group was right or different)
    if std_r > 1e-7:
        loss.backward()
        torch.nn.utils.clip_grad_norm_(rlvr_model.parameters(), 1.0)
        optimizer_rlvr.step()
    
    # 7. Logging & Debugging
    if step % 10 == 0:
        print(f"\nStep {step} | Mean Reward: {mean_r.item()*100}% | Loss: {loss.item():.4f}")
        # DEBUG PRINT: See if the model is actually thinking
        print(f"Sample Thought: {gen_texts[0][:120]}...")

# Save result
rlvr_model.save_pretrained("rlvr_policy_lora")
print("\n✅ Task C6 Complete: RLVR Model saved!")

# Memory Purge
del rlvr_model, base_rlvr, optimizer_rlvr
gc.collect(); torch.cuda.empty_cache()

Loading SFT for RLVR starting point...
Loading HuggingFaceTB/SmolLM2-1.7B-Instruct...


Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]


Step 0 | Mean Reward: 50.0% | Loss: 21.5901
Sample Thought:  Natalia sold 48 clips to her friends in April. In May, she sold half as many clips as she did in April. To find out how...

Step 10 | Mean Reward: 0.0% | Loss: -0.0000
Sample Thought:  First, we need to find out how many people were on the ship the monster ate in the first hundred years. We know that ov...

Step 20 | Mean Reward: 25.0% | Loss: 39.2104
Sample Thought:  First, we know that Bella bought 11 snowflake stamps. Second, she bought 9 more truck stamps than snowflake stamps. So,...

Step 30 | Mean Reward: 0.0% | Loss: -0.0000
Sample Thought:  First, we need to find out how many pizza pieces each person eats. Bill and Dale eat 50% of their pizzas, which means t...

Step 40 | Mean Reward: 0.0% | Loss: -0.0000
Sample Thought:  Mr. Benson bought 12 tickets and received a 5% discount for every ticket bought that exceeds 10. So, he bought 2 ticket...

Step 50 | Mean Reward: 0.0% | Loss: -0.0000
Sample Thought:  Gerald spend

/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:309: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(



✅ Task C6 Complete: RLVR Model saved!


In [16]:
# Cell 20-23: THE FINAL RESET EVALUATION (FORCING UNIQUE WEIGHTS)
import pandas as pd
import numpy as np
import os
import torch
from peft import PeftModel

def run_actual_evaluation():
    print("🚀 Forcing a Hard Reset of all models for final report...")
    working_dir = "/kaggle/working"
    
    # 1. Define paths and verify folders exist
    methods = {
        "SFT":  os.path.join(working_dir, "sft_policy_lora"),
        "PPO":  os.path.join(working_dir, "ppo_policy_lora"),
        "DPO":  os.path.join(working_dir, "dpo_policy_lora"),
        "GRPO": os.path.join(working_dir, "grpo_policy_lora"),
        "RLVR": os.path.join(working_dir, "rlvr_policy_lora")
    }

    # 2. Load the Reward Model (Judge) once
    print("Loading Reward Judge...")
    base_rm, rm_tok = load_base_model(REWARD_ID, is_reward=True)
    judge_rm = PeftModel.from_pretrained(base_rm, os.path.join(working_dir, "reward_model_lora"))
    judge_rm.eval()

    # 3. Load one SINGLE Base Policy Model
    print("Loading Base Policy Model...")
    # Use load_base_model but we will manually attach adapters
    base_policy, p_tok = load_base_model(POLICY_ID)
    
    # Wrap base_policy as a PeftModel starting with SFT
    model = PeftModel.from_pretrained(base_policy, methods["SFT"], adapter_name="SFT")
    
    # Now load the others into the SAME model object but give them unique names
    for name, path in methods.items():
        if name != "SFT" and os.path.exists(path):
            print(f"Adding adapter: {name}")
            model.load_adapter(path, adapter_name=name)

    # 4. Evaluation Logic
    test_prompts = [val_data[i]['prompt'] for i in range(10)]
    final_report = []
    qualitative_samples = {}

    for name in methods.keys():
        if name not in model.peft_config: continue
        
        print(f"--- Running Inference for {name} ---")
        # CRITICAL: Force the model to use the specific weights
        model.set_adapter(name)
        model.eval()

        scores = []
        for i, prompt in enumerate(test_prompts):
            # Generate (Strict Greedy)
            inputs = p_tok(prompt, return_tensors="pt").to(device)
            with torch.no_grad():
                # We use max_new_tokens=48 to ensure differences manifest early
                out = model.generate(**inputs, max_new_tokens=48, do_sample=False, temperature=1.0)
            
            resp = p_tok.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
            
            # Score with Judge
            rm_in = rm_tok(prompt + resp, return_tensors="pt").to(device)
            with torch.no_grad():
                scores.append(judge_rm(**rm_in).logits.item())
            
            if i == 0: qualitative_samples[name] = resp

        final_report.append({
            "Method": name, 
            "Avg RM Score": round(np.mean(scores), 4)
        })

    # 5. Output Final Results
    print("\n" + "="*50)
    print("      TRUE ALIGNMENT PROJECT REPORT")
    print("="*50)
    df = pd.DataFrame(final_report)
    sft_val = df[df['Method'] == "SFT"]["Avg RM Score"].values[0]
    df["Gain vs SFT"] = df["Avg RM Score"] - sft_val
    print(df)
    
    print("\n--- QUALITATIVE CHECK ---")
    for name, txt in qualitative_samples.items():
        print(f"\n[{name}]: {txt[:120]}...")

# Execute
run_actual_evaluation()

🚀 Forcing a Hard Reset of all models for final report...
Loading Reward Judge...
Loading /kaggle/input/datasets/karan943/meta-llamallama-3-2-1b-instruct/meta-llama/Llama-3.2-1B-Instruct...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

LlamaForSequenceClassification LOAD REPORT from: /kaggle/input/datasets/karan943/meta-llamallama-3-2-1b-instruct/meta-llama/Llama-3.2-1B-Instruct
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading Base Policy Model...
Loading HuggingFaceTB/SmolLM2-1.7B-Instruct...


...

special_tokens_map.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Adding adapter: PPO
Adding adapter: DPO
Adding adapter: GRPO
Adding adapter: RLVR
--- Running Inference for SFT ---
--- Running Inference for PPO ---
--- Running Inference for DPO ---
--- Running Inference for GRPO ---
--- Running Inference for RLVR ---

      TRUE ALIGNMENT PROJECT REPORT
  Method  Avg RM Score  Gain vs SFT
0    SFT        0.2329       0.0000
1    PPO       -0.0579      -0.2908
2    DPO        0.6840       0.4511
3   GRPO        0.1811      -0.0518
4   RLVR        0.1385      -0.0944

--- QUALITATIVE CHECK ---

[SFT]:  No.  You can’t write on your hand.  You can’t write on anything.  You can’t even write on your own body.  You can’t wri...

[PPO]:  No.  You can’t write on your hand.  You can’t write on anything.  You can’t even write on your own body.  You can’t wri...

[DPO]:  No.  You can’t write on your hand.  You can’t write on anything.  You can’t even write on your shirt.  You can’t write ...

[GRPO]:  No.  You can’t write on your hand.  You can’t write on anyth